# Exercise — Verify Cross-Domain Reporting Traceability

**Trailhead Provisions**' CSRD quarterly carbon disclosure (`quarterly_sustainability`) spans
**E-commerce Orders** (`orders`) and **Inventory & Fulfillment** (`shipment`), requiring two central
cross-domain rules:
1. **Unit of measure:** `shipment.carbon_kg` must be in kilograms (`kg`), whereas ~3% of fulfillment
   rows are stored in grams (`g`, scaled 1000×).
2. **Quarter attribution timestamp:** Orders has `orders.order_ts` while Inventory has
   `shipment.ship_ts` (1–5 days later, which can cross a quarter boundary); the central standard
   attributes emissions to the order quarter via `orders.order_ts` (excluding `status == 'cancelled'`).

Run the pipeline, trace the report's upstream lineage and a disclosed quarter back to source, and
quantify where integrity breaks. See `INSTRUCTIONS.md`.

## Step 0: Connect to your Cloud Lab

Run the cell below to configure your live AWS environment and ensure the `trailhead` Glue Data Catalog and S3 tables are seeded in your lab account.

In [ ]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
# Seed the Glue Data Catalog and S3 tables if not yet present in the lab account
glue = boto3.client("glue", region_name=os.environ["CARDINAL_AWS_REGION"])
try:
    glue.get_database(Name="trailhead")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

In [ ]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog()
print(f"Backend: {lf_backend()} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")

upstream = gc.lineage.trace_upstream("quarterly_sustainability")
print("Upstream of quarterly_sustainability:", upstream)

rep = gc.run_sustainability_report()
display(rep.quarterly)
rep.reconciliation

## 1. Trace one quarter to source
Pick a reported quarter (e.g. `2023Q4`), trace it to source shipments in `rep.per_order` (`order_carbon`), and isolate the rows causing the disclosed-vs-recomputed gap (`carbon_kg > 100`). Also inspect the five-quarter total inflation across `rep.reconciliation`.

In [ ]:
# TODO(you): pick a quarter (e.g. rep.quarterly.iloc[0]["quarter"]), trace it to rep.per_order,
# and isolate the rows responsible for the disclosed-vs-recomputed discrepancy (hint: carbon_kg > 100 is grams).
q0 = None
detail = None
suspect = None
print(q0, detail, suspect)

## 2. Traceability assessment
Replace the cell below with your assessment.

> **Your traceability assessment here.** Address:
>
> - The **shared carbon metric definition** and why the two cross-domain choices (**unit of measure**: `kg` vs `g` on `shipment.carbon_kg`; **quarter attribution timestamp**: `orders.order_ts` vs `shipment.ship_ts`, excluding cancelled orders) must be standardized centrally.
> - **Where traceability holds** (`quarterly_sustainability` / `rep.quarterly` ← `order_carbon` / `rep.per_order` ← `orders` + `shipment` via `gc.lineage.trace_upstream("quarterly_sustainability")`).
> - **Where integrity breaks** (the unit defect), **quantified** from `rep.reconciliation` for both your traced quarter (`2023Q4`) and the five-quarter total (`2023Q4`–`2024Q4`).
> - The **fix** (unit normalization + pre-disclosure monitor and reconciliation gate).